The AB test is a randomized experiment that is used by most large companies to evaluate the launch of new functionality. Various difficulties may arise in the analysis and conduct of the experiment. Several typical problematic cases from real life are given in this dataset and analysis.

### Content
1. [Loading and processing data](#Loading)
2. [Statistical Analysis](#Statistical)
3. [Conclusions](#Conclusions)

### Loading and processing data <a name="Loading"/>

Let's import the necessary libraries and load the data

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import seaborn as sns
from scipy.stats import pearsonr
from scipy.stats import shapiro

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set(style="whitegrid")

# Load the dataset (expects AB_Test_Results.csv in the working directory)
# Fallback to /mnt/data when running in this environment
import os
candidates = ["AB_Test_Results.csv", "/mnt/data/AB_Test_Results.csv"]
for path in candidates:
    if os.path.exists(path):
        DATA_PATH = path
        break
else:
    raise FileNotFoundError("AB_Test_Results.csv not found in working dir or /mnt/data")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "shape=", df.shape)
df.head()


In [ ]:

display(df.head())
print("\nInfo:")
print(df.info())
print("\nDescribe:")
display(df.describe(include='all'))
df['VARIANT_NAME'].value_counts()


Let's see if there are users who have two A/B test groups

In [ ]:

grp_counts = df.groupby('USER_ID')['VARIANT_NAME'].nunique()
dist_counts = grp_counts.value_counts().sort_index()
print("Unique groups per user (counts):\n", dist_counts)
dist_counts.plot(kind='bar', figsize=(6,4), title='Unique A/B groups per user')
plt.xlabel('Number of unique groups for a user'); plt.ylabel('User count'); plt.show()


In [ ]:

rel = grp_counts.value_counts(normalize=True).sort_index()
print("Relative distribution of unique groups per user:\n", rel.round(4))
(rel*100).plot(kind='bar', figsize=(6,4), title='Unique A/B groups per user (percent)')
plt.xlabel('Number of unique groups for a user'); plt.ylabel('Percent'); plt.show()


What can you observe ? Should we exclude these users ? Why ?

In [ ]:

valid_users = grp_counts[grp_counts == 1].index
ratio = len(valid_users) / grp_counts.shape[0]
print(f"Users with exactly one A/B group: {len(valid_users)} / {grp_counts.shape[0]} ({ratio:.2%})")

df_onegroup = df[df['USER_ID'].isin(valid_users)].copy()
print("Filtered shape:", df_onegroup.shape)
df_onegroup.head()


In [ ]:

# We already limited to users who appear in exactly one variant.
# Double-check that after filtering, each user has exactly one variant.
check = df_onegroup.groupby('USER_ID')['VARIANT_NAME'].nunique().value_counts().to_dict()
print("Post-filter unique variants per user distribution:", check)
assert list(check.keys()) == [1], "Some users still have >1 variants after filtering"
print("User count (post-filter):", df_onegroup['USER_ID'].nunique())


Let's see how the data is distributed

In [ ]:

plt.figure(figsize=(8,5))
sns.boxplot(data=df_onegroup, x='VARIANT_NAME', y='REVENUE', showfliers=True)
plt.title('Revenue distribution by variant (raw)')
plt.show()

plt.figure(figsize=(8,5))
sns.violinplot(data=df_onegroup, x='VARIANT_NAME', y='REVENUE', cut=0, inner='quartile')
plt.title('Revenue distribution by variant (violin)')
plt.show()


It can be seen that there is a strong outlier in the data - we will find it by sorting these revenue values in descending order

In [ ]:

df_onegroup.sort_values('REVENUE', ascending=False).head(10)


In [ ]:

outlier_row = df_onegroup.sort_values('REVENUE', ascending=False).iloc[0]
outlier_user = int(outlier_row['USER_ID'])
print("Top outlier user:", outlier_user, "row:", dict(outlier_row))

df_onegroup[df_onegroup['USER_ID'] == outlier_user].sort_values('REVENUE', ascending=False)


We see that there is only one outlier - in conditions of limited information, we will remove this entry and look at the distribution of data again

In [ ]:

# Remove the single max revenue record (to reduce skew)
max_rev = df_onegroup['REVENUE'].max()
df_trim = df_onegroup[df_onegroup['REVENUE'] < max_rev].copy()
print("Removed the top outlier value:", max_rev, "New shape:", df_trim.shape)

plt.figure(figsize=(8,5))
sns.boxplot(data=df_trim, x='VARIANT_NAME', y='REVENUE', showfliers=True)
plt.title('Revenue distribution by variant (after removing top outlier)')
plt.show()



What can you say about the users and their purchases ?

In [ ]:

g = df_trim.groupby('USER_ID')['REVENUE']
has_zero = g.apply(lambda s: (s == 0).any())
has_pos  = g.apply(lambda s: (s > 0).any())
both = (has_zero & has_pos)
print("Users with both zero & positive revenue records:", both.sum())
both[both].head()


Can a user have records with both zero and positive revenue ?

Let's make the assumption that the records are user visits to the service, and the experimental unit is users.


In [ ]:

df_user = (df_trim
           .groupby(['USER_ID','VARIANT_NAME'], as_index=False)
           .agg(total_revenue=('REVENUE','sum'),
                orders=('REVENUE', lambda s: (s>0).sum())))
print("User-level shape:", df_user.shape)
df_user.head()


In [ ]:

plt.figure(figsize=(8,5))
sns.boxplot(data=df_user, x='VARIANT_NAME', y='total_revenue', showfliers=True)
plt.title('User-level total revenue by variant')
plt.show()


It should be noted that during the transformation, the quantiles in the test group increased

It can be seen that in the test group, almost all quantile statistics, except for the minimum, are at a slightly lower level.

Let's look at various statistics in the context of AB test groups for all users

In [ ]:

def group_metrics(df_user):
    dfm = df_user.copy()
    dfm['is_payer'] = dfm['total_revenue'] > 0
    metrics = (dfm.groupby('VARIANT_NAME')
               .agg(users=('USER_ID','nunique'),
                    payers=('is_payer','sum'),
                    conversion_rate=('is_payer','mean'),
                    arpu=('total_revenue','mean'),
                    median_revenue=('total_revenue','median'),
                    orders_per_user=('orders','mean')))
    # ARPPU: average revenue among payers
    arppu = (dfm[dfm['is_payer']]
             .groupby('VARIANT_NAME')['total_revenue'].mean()
             .rename('arppu'))
    metrics = metrics.join(arppu, how='left').fillna(0)
    return metrics

metrics_all = group_metrics(df_user)
print("Metrics (all users):")
display(metrics_all)


What can you see in the test group about the total amount of revenue, the average check per user, and the number of orders per user slightly increased ?

Let's also see how paying users behave :

In [ ]:

df_user_paid = df_user[df_user['total_revenue'] > 0].copy()
metrics_paid = (df_user_paid.groupby('VARIANT_NAME')
                .agg(payers=('USER_ID','nunique'),
                     arppu=('total_revenue','mean'),
                     median_paid=('total_revenue','median'),
                     orders_per_payer=('orders','mean')))
print("Metrics (paying users only):")
display(metrics_paid)


Let's look at the distributions of all and only paying users

In [ ]:
f, axes = plt.subplots(2, figsize=(10,8))
# build graphs of distributions of all users
sns.distplot(df.loc[df['VARIANT_NAME'] == 'control', 'REVENUE'], ax = axes[0], label='control')
sns.distplot(df.loc[df['VARIANT_NAME'] == 'variant', 'REVENUE'], ax = axes[0], label='variant')
axes[0].set_title('Distribution of revenue of all users')

# build graphs of distributions of paying users
sns.distplot(df.loc[(df['VARIANT_NAME'] == 'control') & (df['REVENUE'] > 0), 'REVENUE'], ax = axes[1], label='control' )
sns.distplot(df.loc[(df['VARIANT_NAME'] == 'variant') & (df['REVENUE'] > 0), 'REVENUE'], ax = axes[1], label='variant' )
axes[1].set_title('Paying user revenue distribution')
plt.legend()
plt.subplots_adjust(hspace = 0.3)

### Statistical Analysis <a name="Statistical"/>

#### Checking if the distribution is normal

Based on their previous graph, we see that the data is not normally distributed.

In [ ]:

from scipy.stats import shapiro

for group, arr in df_user.groupby('VARIANT_NAME')['total_revenue']:
    sample = arr.values
    n = min(len(sample), 5000)  # Shapiro limit / performance
    stat, p = shapiro(sample[:n])
    print(f"Shapiro-Wilk for {group}: W={stat:.4f}, p={p:.4g}, n={n}")
    print("Reject normality?" , "Yes" if p < 0.05 else "No")


Is the null hypothesis about the normal distribution of the data rejected ?

#### Mann-Whitney test

Let's check the value of the statistics of the Mann-Whitney test. Some sources have a limitation of applicability in case of duplicate data. There are a lot of repetitions in our sample, and especially a lot of zero values, so in this case we need to be careful about this criterion.

In [ ]:
(df['REVENUE'] == 0).value_counts()

In [ ]:

from scipy.stats import mannwhitneyu
a = df_user[df_user['VARIANT_NAME'].str.lower().str.contains('control')]['total_revenue'].values
b = df_user[~df_user['VARIANT_NAME'].str.lower().str.contains('control')]['total_revenue'].values
u_stat, p_val = mannwhitneyu(a, b, alternative='two-sided')
print(f"Mann-Whitney U (all users): U={u_stat:.0f}, p={p_val:.6f}")


In [ ]:

from scipy.stats import mannwhitneyu
a_paid = df_user[(df_user['total_revenue']>0) & (df_user['VARIANT_NAME'].str.lower().str.contains('control'))]['total_revenue'].values
b_paid = df_user[(df_user['total_revenue']>0) & (~df_user['VARIANT_NAME'].str.lower().str.contains('control'))]['total_revenue'].values
u_stat_p, p_val_p = mannwhitneyu(a_paid, b_paid, alternative='two-sided')
print(f"Mann-Whitney U (paying users): U={u_stat_p:.0f}, p={p_val_p:.6f}")


#### Bootstrap

In order to get more complete information about the differences between the average values of the ab test groups, we will use bootstap.

Let's create a function to get back samples and get a confidence interval, and then look at the sample statistics

In [ ]:

import numpy as np

def get_bootstrap_samples(data, n_samples=2000, sample_size=None, random_state=42):
    rng = np.random.default_rng(random_state)
    data = np.asarray(data)
    if sample_size is None:
        sample_size = len(data)
    # Return matrix (n_samples x sample_size) of bootstrap samples
    idx = rng.integers(0, len(data), size=(n_samples, sample_size))
    return data[idx]


In [ ]:

control_all = df_user[df_user['VARIANT_NAME'].str.lower().str.contains('control')]['total_revenue'].values
variant_all = df_user[~df_user['VARIANT_NAME'].str.lower().str.contains('control')]['total_revenue'].values

control_bs = get_bootstrap_samples(control_all, n_samples=3000)
variant_bs = get_bootstrap_samples(variant_all, n_samples=3000)

# Precompute means for speed
control_means = control_bs.mean(axis=1)
variant_means = variant_bs.mean(axis=1)
print("Bootstrap (all users) -> control/variant means ready:", control_means.shape, variant_means.shape)


In [ ]:

control_paid = df_user[(df_user['VARIANT_NAME'].str.lower().str.contains('control')) & (df_user['total_revenue']>0)]['total_revenue'].values
variant_paid_arr = df_user[(~df_user['VARIANT_NAME'].str.lower().str.contains('control')) & (df_user['total_revenue']>0)]['total_revenue'].values

control_paid_bs = get_bootstrap_samples(control_paid, n_samples=3000)
variant_paid_bs = get_bootstrap_samples(variant_paid_arr, n_samples=3000)

control_paid_means = control_paid_bs.mean(axis=1)
variant_paid_means = variant_paid_bs.mean(axis=1)
print("Bootstrap (paying users) -> control/variant means ready:", control_paid_means.shape, variant_paid_means.shape)


Let's look at the distribution of means in the ab test groups

In [ ]:
f, ax = plt.subplots()
# plt.figure(figsize=(20,5))
sns.kdeplot(np.mean(control, axis=1), shade=True, label='control')
sns.kdeplot(np.mean(variant, axis=1), shade=True, label='variant')
plt.title('Sample mean distribution for all users')

In [ ]:
f, ax = plt.subplots()
# plt.figure(figsize=(20,5))
sns.kdeplot(np.mean(control_paid, axis=1), shade=True, label='control')
sns.kdeplot(np.mean(variant_paid, axis=1), shade=True, label='variant')
plt.title('Sample mean distribution for paying users')

Do you see any difference ? What about the confidence intervals ? Conclude.

Let's evaluate the difference between the groups: look at the distribution of the mean difference and build confidence intervals for it. To do this, we will create a function for visualization

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

def plot_distribution_and_stat_intervals(variant, control, title, alpha=0.05):
    diff = np.array(variant) - np.array(control)
    lo, hi = np.percentile(diff, [100*alpha/2, 100*(1-alpha/2)])
    mean_diff = diff.mean()

    plt.figure(figsize=(8,5))
    plt.hist(diff, bins=50, density=True)
    plt.title(f"{title}\nMean diff={mean_diff:.4f}, {int((1-alpha)*100)}% CI: [{lo:.4f}, {hi:.4f}]")
    plt.xlabel("Variant mean - Control mean")
    plt.ylabel("Density")
    plt.show()

    print(f"{title}: mean diff={mean_diff:.6f}, {int((1-alpha)*100)}% CI: [{lo:.6f}, {hi:.6f}]")


Let's build a graph of the distribution of the difference in the means and get a confidence interval

For all users

In [ ]:
plot_distribution_and_stat_intervals(np.mean(variant, axis=1),
                                     np.mean(control, axis=1),
                                     title='all users')

In [ ]:
plot_distribution_and_stat_intervals(np.mean(variant_paid, axis=1),
                                     np.mean(control_paid, axis=1),
                                     title='paying users')

What can you observe ? Conclude about the statistical change in average revenue between A/B test groups.
